In [13]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse

# 1. LOAD CLEANED DATA
# Ensure your cleaned_data.csv has InvoiceNo, StockCode, and Description
df = pd.read_csv("C:/Users/FINREG/smart-ecommerce-recommender/data/online_retail.csv")

def build_similarity_matrix(df):
    """
    Creates a product-to-product similarity matrix.
    """
    # Create a Pivot Table: Rows = Invoices, Columns = Products
    # We use 'Quantity' but fill with 1 to indicate presence in the basket
    customer_item_matrix = df.pivot_table(
        index='InvoiceNo', 
        columns='StockCode', 
        values='Quantity',
        aggfunc='sum'
    ).fillna(0)

    # Convert values to binary (1 if bought, 0 otherwise)
    # This reduces bias from bulk buyers
    customer_item_matrix = customer_item_matrix.map(lambda x: 1 if x > 0 else 0)

    # Transpose the matrix to get Item-to-Item relationships
    item_item_matrix = customer_item_matrix.T
    
    # Calculate Cosine Similarity
    # This computes the similarity score between all product vectors
    item_similarity = cosine_similarity(item_item_matrix)
    
    # Convert back to a DataFrame for easy lookup
    recommender_df = pd.DataFrame(
        item_similarity, 
        index=item_item_matrix.index, 
        columns=item_item_matrix.index
    )
    
    return recommender_df

# --- 1. CRÉATION DU LOOKUP (Dictionnaire de correspondance) ---
# On utilise groupby pour s'assurer qu'un StockCode = UNE seule description.
# Cela transforme product_lookup en une "Series" Pandas simple.
product_lookup = df.groupby('StockCode')['Description'].first()

# --- 2. FONCTION DE RECOMMANDATION CORRIGÉE ---
def get_recommendations(stock_code, recommender_df, top_n=5):
    try:
        # On récupère les scores de similarité pour ce produit
        # On trie du plus haut au plus bas et on enlève le produit lui-même
        scores = recommender_df[stock_code].sort_values(ascending=False).drop(stock_code)
        
        # On sélectionne les top N codes produits (les index)
        top_ids = scores.head(top_n).index
        
        # On crée un petit tableau (DataFrame) avec les noms et les scores
        results = pd.DataFrame({
            'StockCode': top_ids,
            'Description': [product_lookup.get(idx, "Unknown") for idx in top_ids],
            'Similarity': scores.head(top_n).values
        })
        
        return results
    except KeyError:
        return "Product ID not found in matrix."

# --- 3. EXÉCUTION ---
print("Building similarity matrix... please wait.")
sim_matrix = build_similarity_matrix(df)

# Exemple d'utilisation
example_id = '85123A' 

# Correction ici : on accède au nom directement car product_lookup est une Series
nom_du_produit = product_lookup.get(example_id, "Unknown Product")

print(f"\nTop recommendations for {nom_du_produit}:")
print(get_recommendations(example_id, sim_matrix))

Building similarity matrix... please wait.

Top recommendations for WHITE HANGING HEART T-LIGHT HOLDER:
  StockCode                        Description  Similarity
0     21733   RED HANGING HEART T-LIGHT HOLDER    0.382730
1     22804    CANDLEHOLDER PINK HANGING HEART    0.328808
2     22470              HEART OF WICKER LARGE    0.239320
3     82482  WOODEN PICTURE FRAME WHITE FINISH    0.237683
4     22457    NATURAL SLATE HEART CHALKBOARD     0.237524
